In [1]:
# Import Required Libraries
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Summary of Preprocessing Steps

## What was done:
1. **Loaded all 5 stock price CSV files** (FPT, HPG, POW, VCB, VIC)
2. **Applied Forward Fill** to handle missing data from holidays and non-trading days
3. **Calculated Daily Returns** (% change) instead of using absolute prices to achieve stationarity
4. **Split data** into 70% training, 20% test, and 10% validation sets **before scaling**
5. **Applied MinMaxScaler** independently to each set to prevent data leakage
6. **Saved preprocessed data** as CSV files and scaler objects as pickle files

## Key Points:
- ✓ Data leakage avoided by splitting first, then scaling each set independently
- ✓ Scaler was fit only on training data, then applied to test and validation
- ✓ Daily returns (pct_change) remove non-stationarity in prices
- ✓ All features normalized to [0, 1] range using MinMaxScaler
- ✓ Original scaler objects saved for future predictions on new data

In [2]:

import pickle

# Create subdirectories for processed data
processed_subdir = os.path.join(processed_dir, 'scaled')
os.makedirs(processed_subdir, exist_ok=True)

print("\n" + "="*80)
print("SAVING PREPROCESSED DATA")
print("="*80 + "\n")

for stock_name, data_dict in scaled_data.items():
    # Create stock-specific subdirectory
    stock_dir = os.path.join(processed_subdir, stock_name)
    os.makedirs(stock_dir, exist_ok=True)
    
    # Save train, test, validation sets as CSV
    train_file = os.path.join(stock_dir, f'{stock_name}_train_scaled.csv')
    test_file = os.path.join(stock_dir, f'{stock_name}_test_scaled.csv')
    val_file = os.path.join(stock_dir, f'{stock_name}_val_scaled.csv')
    
    data_dict['train'].to_csv(train_file, index=False)
    data_dict['test'].to_csv(test_file, index=False)
    data_dict['val'].to_csv(val_file, index=False)
    
    # Save scaler object as pickle file
    scaler_file = os.path.join(stock_dir, f'{stock_name}_scaler.pkl')
    with open(scaler_file, 'wb') as f:
        pickle.dump(data_dict['scaler'], f)
    
    print(f"Stock: {stock_name}")
    print(f"  ✓ Train set saved: {train_file}")
    print(f"  ✓ Test set saved: {test_file}")
    print(f"  ✓ Validation set saved: {val_file}")
    print(f"  ✓ Scaler saved: {scaler_file}")
    print()

print("="*80)
print("DATA PREPROCESSING COMPLETED SUCCESSFULLY!")
print("="*80)

NameError: name 'processed_dir' is not defined

# 6. Save Preprocessed Data
Export the scaled and preprocessed datasets (train, test, validation) for each stock to CSV files for downstream analysis.

In [ ]:

# Apply MinMaxScaler to scale features to [0, 1]
# Important: Fit on training data only, then transform test and validation
scaled_data = {}

for stock_name, split_dict in splits.items():
    # Get feature columns (exclude 'time' if present)
    if 'time' in split_dict['train'].columns:
        feature_cols = [col for col in split_dict['train'].columns if col != 'time']
    else:
        feature_cols = [col for col in split_dict['train'].columns]
    
    train_df = split_dict['train']
    test_df = split_dict['test']
    val_df = split_dict['val']
    
    # Initialize MinMaxScaler
    scaler = MinMaxScaler()
    
    # Fit scaler on training data
    train_scaled = train_df.copy()
    train_scaled[feature_cols] = scaler.fit_transform(train_df[feature_cols])
    
    # Transform test and validation data using the same scaler
    test_scaled = test_df.copy()
    test_scaled[feature_cols] = scaler.transform(test_df[feature_cols])
    
    val_scaled = val_df.copy()
    val_scaled[feature_cols] = scaler.transform(val_df[feature_cols])
    
    scaled_data[stock_name] = {
        'train': train_scaled,
        'test': test_scaled,
        'val': val_scaled,
        'scaler': scaler,
        'feature_columns': feature_cols
    }
    
    print(f"\n{'='*60}")
    print(f"Stock: {stock_name} - Data Scaling (MinMaxScaler)")
    print(f"{'='*60}")
    print(f"Training data scaled - Shape: {train_scaled.shape}")
    print(f"Test data scaled - Shape: {test_scaled.shape}")
    print(f"Validation data scaled - Shape: {val_scaled.shape}")
    print(f"\nTrain set statistics after scaling:")
    print(train_scaled[feature_cols].describe().round(4))

# 5. Apply Data Scaling (MinMaxScaler)
Scale each train, test, and validation set independently to prevent data leakage. Fit scaler on train set, then apply to test and validation sets.

In [ ]:

# Split data: 70% train, 20% test, 10% validation
splits = {}
feature_columns = [col for col in returns_data[stock_name].columns if col != 'time']

for stock_name, df in returns_data.items():
    # Drop NaN values created by pct_change
    df_clean = df.dropna().reset_index(drop=True)
    
    # First split: 70% train, 30% temp (for test + validation)
    train_df, temp_df = train_test_split(df_clean, test_size=0.30, shuffle=False)
    
    # Second split: Split 30% into 20% test and 10% validation
    # Of the 30%, we need 20/30 for test and 10/30 for validation
    test_df, val_df = train_test_split(temp_df, test_size=0.333, shuffle=False)  # 10/30 ≈ 0.333
    
    splits[stock_name] = {
        'train': train_df.reset_index(drop=True),
        'test': test_df.reset_index(drop=True),
        'val': val_df.reset_index(drop=True)
    }
    
    total_rows = len(df_clean)
    print(f"\n{'='*60}")
    print(f"Stock: {stock_name} - Data Split")
    print(f"{'='*60}")
    print(f"Total rows: {total_rows}")
    print(f"Train set: {len(train_df)} rows ({len(train_df)/total_rows*100:.1f}%)")
    print(f"Test set:  {len(test_df)} rows ({len(test_df)/total_rows*100:.1f}%)")
    print(f"Validation set: {len(val_df)} rows ({len(val_df)/total_rows*100:.1f}%)")

# 4. Split Data into Train (70%), Test (20%), and Validation (10%) Sets
Divide each preprocessed dataset BEFORE applying any scaling to prevent data leakage. Scaling will be applied separately to each set.

In [ ]:

# Calculate daily returns for price columns
price_columns = ['open', 'close', 'low', 'high']
returns_data = {}

for stock_name, df in stock_data.items():
    df_returns = df.copy()
    
    # Calculate daily returns (% change) for price columns
    for col in price_columns:
        df_returns[f'{col}_return'] = df[col].pct_change()
    
    # Remove the original price columns and keep only returns and time
    df_returns = df_returns[['time'] + [f'{col}_return' for col in price_columns] + ['volume']]
    
    # Drop the first row (NaN from pct_change)
    df_returns = df_returns.iloc[1:].reset_index(drop=True)
    
    returns_data[stock_name] = df_returns
    
    print(f"\n{'='*60}")
    print(f"Stock: {stock_name} - Daily Returns Calculated")
    print(f"{'='*60}")
    print(f"Shape: {df_returns.shape}")
    print(f"\nFirst few rows:")
    print(df_returns.head())
    print(f"\nDaily returns statistics:")
    print(df_returns.describe())

# 3. Calculate Daily Returns
Transform price features by calculating percentage change (daily return) to remove non-stationary characteristics instead of using absolute prices.

In [ ]:

# Fill missing values with forward fill
final_missing = {}
missing_report = []

for stock_name, df in stock_data.items():
    # Count missing values before
    missing_before = df.isnull().sum().sum()
    missing_dates_before = df[df.isnull().any(axis=1)].shape[0]
    
    # Apply forward fill
    df_filled = df.copy()
    df_filled = df_filled.fillna(method='ffill')
    
    # Fill remaining NaN (if any at the start)
    df_filled = df_filled.fillna(method='bfill')
    
    # Count missing values after
    missing_after = df_filled.isnull().sum().sum()
    missing_dates_after = df_filled[df_filled.isnull().any(axis=1)].shape[0]
    
    final_missing[stock_name] = df_filled.isnull().sum()
    stock_data[stock_name] = df_filled
    
    missing_report.append({
        'Stock': stock_name,
        'Missing Values Before': missing_before,
        'Missing Rows Before': missing_dates_before,
        'Missing Values After': missing_after,
        'Missing Rows After': missing_dates_after,
        'Rows Filled': missing_dates_before - missing_dates_after
    })

# Display missing data report
missing_df = pd.DataFrame(missing_report)
print("=" * 80)
print("MISSING DATA REPORT (Forward Fill)")
print("=" * 80)
print(missing_df.to_string(index=False))
print("\n")

# 2. Handle Missing Data with Forward Fill
Apply Forward Fill (ffill) to handle missing data for holidays and non-trading days. Report missing values before and after filling.

In [ ]:

# Define paths
data_dir = r'../data/raw'
processed_dir = r'../data/processed'

# Create processed directory if it doesn't exist
os.makedirs(processed_dir, exist_ok=True)

# Get list of CSV files
csv_files = [f for f in os.listdir(data_dir) if f.endswith('.csv')]
print(f"Found {len(csv_files)} CSV files: {csv_files}\n")

# Load all CSV files
stock_data = {}
initial_missing = {}

for file in csv_files:
    stock_name = file.replace('.csv', '')
    file_path = os.path.join(data_dir, file)
    df = pd.read_csv(file_path)
    df['time'] = pd.to_datetime(df['time'])
    df = df.sort_values('time').reset_index(drop=True)
    stock_data[stock_name] = df
    
    # Count missing values before processing
    initial_missing[stock_name] = df.isnull().sum()
    
    print(f"\n{'='*60}")
    print(f"Stock: {stock_name}")
    print(f"{'='*60}")
    print(f"Shape: {df.shape}")
    print(f"\nFirst few rows:")
    print(df.head())
    print(f"\nData types:\n{df.dtypes}")
    print(f"\n Missing values before processing:")
    print(initial_missing[stock_name])
    print(f"Total missing values: {df.isnull().sum().sum()}")
    print(f"Date range: {df['time'].min()} to {df['time'].max()}")

# 1. Load and Explore Stock Price Data
Load all 5 CSV files from the folder, examine their structure, shape, and display basic statistics. Report the initial number of missing values before preprocessing.